# 05. Evaluación Diagnóstica y Validación Clínica

**Universidad Nacional Federico Villarreal (UNFV)**  
**Facultad de Ingeniería Electrónica e Informática (FIEI)**  
**Autor:** Jimenez Villalva Franz José  

Evaluación en el conjunto ciego de 2,068 niños: Métricas clínicas (Recall, AUC), Validación Cruzada y Gini.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix
import joblib

DATA_PATH = '../data/processed/anemia_infantil_peru_endes.csv'
df = pd.read_csv(DATA_PATH)
X = df[['Hemoglobina_Observada', 'Altitud_msnm', 'Edad_Meses', 'Sexo', 'Area_Residencia', 'Peso_kg', 'Talla_cm']]
y = df['Anemia']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
scaler = joblib.load('../models/scaler.joblib')
X_test_scaled = scaler.transform(X_test)
X_train_scaled = scaler.transform(X_train)

rf = joblib.load('../models/modelo_rf.joblib')
knn = joblib.load('../models/modelo_knn.joblib')


### Tabla Comparativa de Rendimiento


In [ ]:
metrics = []
for name, m in [('KNN (k=17)', knn), ('Random Forest (100)', rf)]:
    preds = m.predict(X_test_scaled)
    proba = m.predict_proba(X_test_scaled)[:, 1]
    metrics.append({
        'Modelo': name,
        'Accuracy': f'{accuracy_score(y_test, preds)*100:.2f}%',
        'Precision': f'{precision_score(y_test, preds)*100:.2f}%',
        'Recall (Sensibilidad)': f'{recall_score(y_test, preds)*100:.2f}%',
        'F1-Score': f'{f1_score(y_test, preds)*100:.2f}%',
        'ROC-AUC': f'{roc_auc_score(y_test, proba):.4f}'
    })
pd.DataFrame(metrics)


### Simulación de Inferencia Pediátrica en Tiempo Real


In [ ]:
# Caso: Infante de 24 meses en Puno (3,825 msnm) con Hb 11.8 g/dL
caso = np.array([[11.8, 3825, 24, 1, 1, 11.2, 81.0]])
caso_esc = scaler.transform(caso)
pred = rf.predict(caso_esc)[0]
prob = rf.predict_proba(caso_esc)[0][1]
print(f'Diagnóstico predicho: {"ANEMIA INFANTIL" if pred == 1 else "NORMAL / SANO"}')
print(f'Probabilidad clínica: {prob*100:.2f}%')
